# NEXTBUY - From Raw Data to Smart Decisions

---

## Contexte

Dans un environnement où les entreprises collectent des volumes massifs de données transactionnelles, la capacité à transformer ces données en décisions stratégiques constitue un avantage compétitif majeur.

Nous disposons d’un ensemble de données comprenant :

* Des millions de commandes
* Des milliers de clients
* Des produits organisés en rayons
* Des rayons regroupés en départements

Cette structure hiérarchique permet une analyse multi-niveaux :
produit, catégorie, client et temporalité.

## Objectifs du Projet

---

Notre démarche repose sur deux axes principaux :

### Analyse Exploratoire des Données (EDA)

Identifier des insights exploitables permettant :

* D’optimiser la performance commerciale
* D’améliorer la fidélisation client
* De comprendre les comportements d’achat

### Modélisation Prédictive

Construire des modèles capables de :

* Prédire la probabilité de réachat d’un produit
* Anticiper la taille du panier d’un client

# Business Questions & Analytical Objectives

---

Afin de structurer notre analyse, nous avons défini les huit questions stratégiques suivantes :

### Quels produits contribuent le plus à la performance globale (volume d’achat et fréquence de reorder) ?

In [ ]:
import pandas as pd

orders = pd.read_csv("data/orders.csv")
order_products = pd.read_csv("data/order_products.csv")
products = pd.read_csv("data/products.csv")

df = orders.merge(order_products, on="order_id")
df = df.merge(products, on="product_id")

volume = df.groupby("product_name").size().reset_index(name="total_purchases")
reorder_rate = df.groupby("product_name")["reordered"].mean().reset_index(name="reorder_rate")

performance = volume.merge(reorder_rate, on="product_name")
performance["performance_score"] = performance["total_purchases"] * performance["reorder_rate"]

top_products = performance.sort_values("performance_score", ascending=False).head(10)

top_products

,product_name,total_purchases,reorder_rate,performance_score
3625,Banana,199654,0.843383,168384.747062
3422,Bag of Organic Bananas,160615,0.831747,133590.990480
31519,Organic Strawberries,111975,0.778374,87158.448619
28461,Organic Baby Spinach,101915,0.772612,78740.772612
29908,Organic Hass Avocado,90372,0.795751,71913.591502
28425,Organic Avocado,74961,0.756704,56723.270111
32073,Organic Whole Milk,58210,0.829454,48282.488361
22115,Large Lemon,64862,0.696608,45183.393216
30965,Organic Raspberries,57805,0.768805,44440.768805
42381,Strawberries,60763,0.697480,42381.000000


## Interprétation des résultats

Le tableau final affiche les 10 produits ayant le **score de performance le plus élevé**.

Ce score combine :

* le nombre total d’achats (`total_purchases`)
* la proportion moyenne de reorder (`reorder_rate`)

Un score élevé signifie qu’un produit est à la fois :

* très acheté
* fréquemment racheté

## Justification méthodologique (par rapport au code)

1. Les fichiers sont fusionnés afin d’associer chaque achat à son produit et à son indicateur de reorder.
2. Le volume est calculé avec un `groupby().size()` pour compter toutes les occurrences d’achat.
3. Le taux de reorder est obtenu avec `mean()` sur la variable binaire `reordered` (0/1), ce qui donne directement une proportion.
4. Le score final est construit comme un produit multiplicatif (`volume × reorder_rate`) pour pondérer la fidélité par la popularité.
5. Le tri décroissant permet d’identifier les produits ayant l’impact global le plus fort.

Cette méthode permet d’éviter de privilégier uniquement les produits très populaires ou uniquement les produits très fidèles : elle combine les deux dimensions dans un indicateur unique.

---


### Quels produits présentent le taux de reorder le plus élevé et quels facteurs peuvent l’expliquer ?

In [ ]:
import pandas as pd

orders = pd.read_csv("data/orders.csv")
order_products = pd.read_csv("data/order_products.csv")
products = pd.read_csv("data/products.csv")

df = orders.merge(order_products, on="order_id")
df = df.merge(products, on="product_id")

product_stats = df.groupby("product_name").agg(
    total_purchases=("product_id", "count"),
    reorder_rate=("reordered", "mean"),
    avg_days_between_orders=("days_since_prior_order", "mean"),
    avg_add_to_cart_position=("add_to_cart_order", "mean")
).reset_index()

product_stats = product_stats[product_stats["total_purchases"] > 100]

top_reorder = product_stats.sort_values("reorder_rate", ascending=False).head(10)

top_reorder

,product_name,total_purchases,reorder_rate,avg_days_between_orders,avg_add_to_cart_position
36314,"Purified Water, 9.5pH+",124,0.911290,NaN,3.975806
19823,Homestyle Orange Juice,135,0.881481,NaN,3.614815
23382,Lo-Carb Energy Drink,221,0.864253,NaN,2.945701
18825,Half And Half Ultra Pasteurized,1224,0.862745,NaN,3.812908
29972,Organic Homogenized Whole Milk,1643,0.858186,NaN,4.823494
47569,Wheat Sandwich Bread,416,0.858173,NaN,6.680288
45928,Ultra-Purified Water,585,0.856410,NaN,4.184615
48184,Whole Organic Omega 3 Milk,3802,0.856391,NaN,5.338243
30163,Organic Lactose Free Whole Milk,3557,0.855174,NaN,4.805735
17542,Goat Milk,2177,0.849793,NaN,4.882407


## Interprétation des résultats

Les produits affichés ont le **taux moyen de reorder le plus élevé** parmi ceux ayant un volume suffisant (>100 achats pour éviter le bruit statistique).

Un taux proche de 1 signifie que le produit est presque systématiquement racheté après un premier achat.

Les colonnes supplémentaires permettent d’explorer des facteurs explicatifs :

* `avg_days_between_orders` : indique si le produit est acheté de manière régulière.
* `avg_add_to_cart_position` : une position faible suggère un produit prioritaire ou essentiel.
* `total_purchases` : permet de distinguer un produit niche très fidèle d’un produit massif et fidèle.

## Justification méthodologique

1. Le `groupby` permet d’agréger les métriques par produit.
2. La moyenne de `reordered` donne directement le taux de reorder car la variable est binaire (0/1).
3. Un filtre sur `total_purchases` est appliqué pour éviter qu’un produit avec très peu d’achats mais 100 % de reorder apparaisse artificiellement en tête.
4. L’ajout de variables comportementales (temps entre commandes, position dans le panier) permet d’explorer des corrélations possibles sans encore construire de modèle explicatif.

Cette approche identifie les produits les plus fidèles et fournit des indicateurs permettant d’analyser les mécanismes associés à cette fidélité.

---

### À quels jours et heures les clients passent-ils le plus de commandes, et quels produits dominent ces créneaux ?

In [ ]:
import pandas as pd

orders = pd.read_csv("data/orders.csv")
order_products = pd.read_csv("data/order_products.csv")
products = pd.read_csv("data/products.csv")

df = orders.merge(order_products, on="order_id")
df = df.merge(products, on="product_id")

orders_by_day = df.groupby("order_dow")["order_id"].nunique().reset_index(name="total_orders")
peak_day = orders_by_day.sort_values("total_orders", ascending=False).head(1)

orders_by_hour = df.groupby("order_hour_of_day")["order_id"].nunique().reset_index(name="total_orders")
peak_hour = orders_by_hour.sort_values("total_orders", ascending=False).head(1)

top_products_day = df[df["order_dow"] == peak_day["order_dow"].values[0]] \
    .groupby("product_name").size().reset_index(name="purchases") \
    .sort_values("purchases", ascending=False).head(10)

top_products_hour = df[df["order_hour_of_day"] == peak_hour["order_hour_of_day"].values[0]] \
    .groupby("product_name").size().reset_index(name="purchases") \
    .sort_values("purchases", ascending=False).head(10)

peak_day, peak_hour, top_products_day, top_products_hour

(    order_dow  total_orders
 10       10.0        114975,
     order_hour_of_day  total_orders
 30               30.0        129421,
                  product_name  purchases
 2599                   Banana      17330
 2445   Bag of Organic Bananas      13381
 23128    Organic Strawberries       8962
 20536    Organic Baby Spinach       8353
 21754    Organic Hass Avocado       7479
 20503         Organic Avocado       6019
 15919             Large Lemon       5598
 31135            Strawberries       5219
 16660                   Limes       5077
 23592      Organic Whole Milk       4902,
                  product_name  purchases
 2725                   Banana      17244
 2561   Bag of Organic Bananas      11949
 21730    Organic Baby Spinach       9493
 24329    Organic Strawberries       8322
 16842             Large Lemon       6951
 22949    Organic Hass Avocado       6889
 21699         Organic Avocado       6820
 32735            Strawberries       5880
 17635                   

## Interprétation des résultats

* `peak_day` indique le jour de la semaine générant le plus grand nombre de commandes uniques.
* `peak_hour` identifie l’heure avec la plus forte activité.

Les tableaux `top_products_day` et `top_products_hour` montrent les produits les plus achetés durant ces créneaux.

On observe généralement :

* Un pic en fin de semaine.
* Une concentration des commandes en fin de matinée ou début de soirée.
* Une domination de produits frais ou essentiels pendant ces périodes de forte activité.


## Justification méthodologique

1. Le nombre de commandes est mesuré avec `nunique()` sur `order_id` pour éviter de compter plusieurs fois une même commande contenant plusieurs produits.
2. Le tri décroissant permet d’identifier le jour et l’heure à volume maximal.
3. Une fois les créneaux identifiés, un filtrage conditionnel isole les transactions correspondantes.
4. Un `groupby().size()` permet d’identifier les produits dominants sur ces périodes spécifiques.

Cette méthode permet d’analyser simultanément la dimension temporelle (quand les clients commandent) et la dimension produit (quoi ils achètent à ces moments).

---


### Existe-t-il une relation entre le délai depuis la dernière commande et la probabilité qu’un produit soit reorder ?

In [ ]:
import pandas as pd

orders = pd.read_csv("data/orders.csv")
order_products = pd.read_csv("data/order_products.csv")

df = orders.merge(order_products, on="order_id")

df = df.dropna(subset=["days_since_prior_order"])

relation = df.groupby("days_since_prior_order")["reordered"] \
             .mean() \
             .reset_index(name="reorder_probability")

correlation = df["days_since_prior_order"].corr(df["reordered"])

relation.head(), correlation

(Empty DataFrame
 Columns: [days_since_prior_order, reorder_probability]
 Index: [],
 nan)

## Interprétation des résultats

* Le tableau `relation` montre, pour chaque nombre de jours écoulés depuis la dernière commande, la probabilité moyenne qu’un produit soit reorder.
* La variable `correlation` mesure la relation linéaire globale entre le délai et le reorder.

Si la corrélation est négative, cela signifie que plus le délai augmente, plus la probabilité de reorder diminue.
Si elle est positive, cela suggère que les commandes espacées contiennent proportionnellement plus de produits déjà achetés.
Si elle est proche de zéro, l’effet du délai est faible ou non linéaire.


## Justification méthodologique

1. La fusion permet d’associer à chaque ligne produit le délai depuis la commande précédente.
2. Les valeurs manquantes sont supprimées pour éviter un biais dans le calcul.
3. La moyenne de `reordered` par valeur de `days_since_prior_order` donne directement une probabilité conditionnelle, la variable étant binaire (0/1).
4. Le calcul de corrélation permet d’évaluer l’existence d’un lien global entre les deux variables.

Cette approche combine analyse descriptive et mesure statistique simple pour évaluer l’existence d’une relation.

---

### Quels produits sont fréquemment achetés ensemble ?


In [ ]:
import pandas as pd
from itertools import combinations

order_products = pd.read_csv("data/order_products.csv")
products = pd.read_csv("data/products.csv")

df = order_products.merge(products, on="product_id")

basket = df.groupby("order_id")["product_name"].apply(list)

pair_counts = {}

for products_list in basket:
    unique_products = set(products_list)
    for pair in combinations(sorted(unique_products), 2):
        pair_counts[pair] = pair_counts.get(pair, 0) + 1

pairs_df = pd.DataFrame(
    [(p[0], p[1], c) for p, c in pair_counts.items()],
    columns=["product_1", "product_2", "co_purchase_count"]
)

top_pairs = pairs_df.sort_values("co_purchase_count", ascending=False).head(10)

top_pairs

,product_1,product_2,co_purchase_count
3555,Bag of Organic Bananas,Organic Hass Avocado,26434
4366,Bag of Organic Bananas,Organic Strawberries,26006
2169,Banana,Organic Strawberries,23757
433,Banana,Organic Avocado,22781
434,Banana,Organic Baby Spinach,21557
2796,Bag of Organic Bananas,Organic Baby Spinach,21478
6424,Banana,Strawberries,17471
2163,Banana,Large Lemon,17394
1746,Organic Hass Avocado,Organic Strawberries,17311
1256,Bag of Organic Bananas,Organic Raspberries,17075


## Interprétation des résultats

Le tableau affiche les paires de produits apparaissant le plus souvent dans une même commande.

Un `co_purchase_count` élevé signifie que ces deux produits sont régulièrement présents ensemble dans le panier.

Ces associations reflètent généralement :

* Des produits complémentaires (ex. chips + soda).
* Des produits d’une même catégorie.
* Des habitudes d’achat récurrentes.

## Justification méthodologique

1. Les produits sont regroupés par `order_id` pour reconstruire chaque panier.
2. Les combinaisons de taille 2 sont générées pour chaque panier afin d’identifier toutes les paires possibles.
3. L’utilisation d’un `set` évite de compter deux fois un même produit dans une commande.
4. Un comptage global permet de mesurer la fréquence d’apparition de chaque paire.
5. Le tri décroissant identifie les associations les plus fréquentes.

Cette méthode repose sur une logique de co-occurrence simple permettant d’identifier des relations produit-produit sans modèle probabiliste.

---

### Quels profils clients peuvent être identifiés à partir de leurs comportements d’achat ?

In [9]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

orders = pd.read_csv("data/orders.csv")
order_products = pd.read_csv("data/order_products.csv")

df = orders.merge(order_products, on="order_id")

user_features = df.groupby("user_id").agg(
    total_orders=("order_number", "max"),
    avg_days_between_orders=("days_since_prior_order", "mean"),
    avg_cart_position=("add_to_cart_order", "mean"),
    reorder_rate=("reordered", "mean"),
    avg_hour=("order_hour_of_day", "mean")
).reset_index()

user_features["avg_days_between_orders"] = user_features["avg_days_between_orders"].fillna(0)
user_features = user_features.dropna()

X = user_features.drop("user_id", axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42)
user_features["cluster"] = kmeans.fit_predict(X_scaled)

cluster_profiles = user_features.groupby("cluster").mean()

cluster_profiles

,user_id,total_orders,avg_days_between_orders,avg_cart_position,reorder_rate,avg_hour
cluster,,,,,,
0,103315.237079,1.703861,0.0,5.639001,0.355140,18.692847
1,102990.006729,5.218847,0.0,11.705455,0.533387,13.772242
2,103150.187703,5.413323,0.0,4.983812,0.627175,9.874720
3,103083.689018,5.240135,0.0,5.046719,0.271646,20.420783


## Interprétation des résultats

Chaque cluster représente un groupe de clients ayant des comportements similaires.

Les différences entre clusters peuvent révéler :

* Clients fréquents vs occasionnels (nombre total de commandes).
* Clients réguliers vs irréguliers (intervalle moyen entre commandes).
* Clients fidèles vs explorateurs (taux de reorder).
* Acheteurs matinaux vs nocturnes (heure moyenne de commande).

L’analyse des moyennes par cluster permet de qualifier les profils (ex. client régulier fidèle, client impulsif, client panier important mais peu fréquent).

## Justification méthodologique

1. Les comportements sont agrégés au niveau `user_id` afin d’obtenir des caractéristiques représentatives de chaque client.
2. Les variables sont standardisées pour éviter qu’une métrique à grande échelle domine le clustering.
3. KMeans regroupe les clients en fonction de la distance euclidienne dans l’espace des variables normalisées.
4. Le nombre de clusters est fixé arbitrairement ici à 4 pour obtenir une segmentation exploitable et lisible.
5. Le regroupement final par cluster permet d’interpréter les caractéristiques moyennes de chaque segment.

Cette approche permet d’identifier des profils comportementaux sans variable cible supervisée.

---

### Peut-on prédire si un produit sera reorder lors de la prochaine commande d’un client ?

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

df = pd.read_csv("data/merged_clean.csv")

product_freq = df.groupby("product_id").size()
df["product_frequency"] = df["product_id"].map(product_freq)

product_reorder_rate = df.groupby("product_id")["reordered"].mean()
df["product_reorder_rate"] = df["product_id"].map(product_reorder_rate)

user_freq = df.groupby("user_id").size()
df["user_frequency"] = df["user_id"].map(user_freq)

features = [
    "product_frequency",
    "product_reorder_rate",
    "user_frequency",
    "days_since_prior_order",
    "order_hour_of_day",
    "add_to_cart_order"
]

df = df.dropna(subset=features + ["reordered"])

X = df[features]
y = df["reordered"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

log_accuracy = accuracy_score(y_test, log_model.predict(X_test))
log_auc = roc_auc_score(y_test, log_model.predict_proba(X_test)[:, 1])

rf_accuracy = accuracy_score(y_test, rf_model.predict(X_test))
rf_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])

log_accuracy, log_auc, rf_accuracy, rf_auc

FileNotFoundError: [Errno 2] No such file or directory: 'data/merged_clean.csv'

## Interprétation des résultats

Les métriques retournées sont :

* Accuracy Logistic Regression
* ROC-AUC Logistic Regression
* Accuracy Random Forest
* ROC-AUC Random Forest

Le ROC-AUC est la métrique la plus pertinente ici, car il mesure la capacité du modèle à distinguer les produits reorder vs non reorder indépendamment du seuil de classification.

Si le Random Forest obtient un ROC-AUC significativement supérieur à la régression logistique, cela indique que les comportements de reorder présentent des relations non linéaires.

Un ROC-AUC supérieur à 0.75 indique une capacité prédictive exploitable.
Un score proche de 0.5 indiquerait une absence de pouvoir prédictif.

## Justification méthodologique

1. Les données sont fusionnées pour relier chaque produit à son contexte de commande.
2. Des variables explicatives comportementales sont construites : fréquence produit, fréquence utilisateur, historique de reorder.
3. La variable cible `reordered` étant binaire, une classification supervisée est appropriée.
4. Un split train/test permet d’évaluer la généralisation du modèle.
5. Deux modèles sont comparés :

   * Régression logistique pour un modèle linéaire interprétable.
   * Random Forest pour capturer des interactions complexes.
6. L’évaluation via ROC-AUC permet de mesurer la qualité du classement probabiliste.

Cette approche permet de vérifier empiriquement si le comportement passé permet de prédire un futur reorder.

---

### Peut-on prédire la taille du panier d’un client en fonction de son historique ?

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df = pd.read_csv("data/merged_clean.csv")

user_stats = df.groupby("user_id").agg(
    total_orders=("order_number", "max"),
    avg_days_between_orders=("days_since_prior_order", "mean"),
    reorder_rate=("reordered", "mean"),
    avg_hour=("order_hour_of_day", "mean")
).reset_index()

df = df.merge(user_stats, on="user_id")

features = [
    "total_orders",
    "avg_days_between_orders",
    "reorder_rate",
    "avg_hour"
]

df = df.dropna(subset=features)

X = df[features]
y = df["cart_size"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
lin_pred = lin_model.predict(X_test)

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

(
    mean_absolute_error(y_test, lin_pred),
    r2_score(y_test, lin_pred),
    mean_absolute_error(y_test, rf_pred),
    r2_score(y_test, rf_pred)
)

FileNotFoundError: [Errno 2] No such file or directory: 'data/merged_clean.csv'

## Interprétation des résultats

Les métriques retournées sont :

* MAE Linear Regression
* R² Linear Regression
* MAE Random Forest
* R² Random Forest

Le MAE mesure l’erreur moyenne en nombre d’articles.
Le R² indique la proportion de variance expliquée.

Si le Random Forest présente un MAE plus faible et un R² plus élevé, cela signifie que la taille du panier dépend de relations non linéaires entre les variables comportementales.

Un R² modéré (0.3–0.5) indique que l’historique explique partiellement le panier.
Un R² faible suggère que d’autres variables non présentes influencent fortement la taille du panier.


## Justification méthodologique

1. La taille du panier est calculée par un comptage du nombre de produits par `order_id`.
2. Des variables historiques sont agrégées au niveau utilisateur pour représenter le comportement global.
3. La variable cible étant continue, une régression supervisée est adaptée.
4. Un split train/test permet d’évaluer la capacité de généralisation.
5. Deux modèles sont comparés :

   * Régression linéaire pour tester une relation simple.
   * Random Forest pour capturer des interactions complexes.

Cette approche permet d’évaluer si le comportement passé du client contient un signal prédictif sur la taille future de ses paniers.

---

# Dataset preparation for Machine Learning

---

### Présentation des datasets

Le projet utilise plusieurs fichiers décrivant les commandes, les produits et les catégories.
`orders` contient les informations sur les commandes, `order_products` relie les produits aux commandes et indique si un produit est reorder, et `products`, `aisles` et `departments` décrivent les produits et leurs catégories.

Ces tables sont reliées par `order_id`, `product_id` et `user_id` afin de reconstituer l’historique des achats.


code

# Model 1 : Reorder prediction

---

Le premier modèle vise à prédire si un produit sera racheté lors d’une prochaine commande `(reordered)`. Chaque observation correspond à un produit présent dans une commande. Les variables utilisées décrivent le contexte d’achat, comme l’heure de la commande, le jour de la semaine, le délai depuis la commande précédente et la position du produit dans le panier. Deux modèles sont entraînés et évalués : une régression logistique et un Random Forest Classifier.

In [3]:
import pandas as pd

df = pd.read_csv("data/merged_clean.csv")

features_model1 = [
    "user_id",
    "product_id",
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    "add_to_cart_order",
    "aisle_id",
    "department_id"
]

target_model1 = "reordered"

model_ready_reorder = df[features_model1 + [target_model1]].dropna()

model_ready_reorder.to_csv("data/model_ready_reorder.csv", index=False)

print("Dataset Model 1 shape :", model_ready_reorder.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'data/merged_clean.csv'

## Interprétation des résultats

Chaque observation correspond à un produit présent dans une commande. La variable cible est reordered, qui indique si le produit a déjà été acheté auparavant par le client.

# Model 2 : Cart size prediction

---

Le second modèle vise à prédire la taille du panier d’une commande `(cart_size)`. Dans ce cas, chaque observation correspond à une commande `(order_id)`. Les variables utilisées décrivent les caractéristiques de la commande, notamment le jour, l’heure et le délai depuis la commande précédente. Deux modèles sont utilisés : une régression linéaire et un Random Forest Regressor.

In [1]:
cart_size = df.groupby("order_id").size().reset_index(name="cart_size")

orders_features = df.groupby("order_id").agg({
    "user_id": "first",
    "order_dow": "first",
    "order_hour_of_day": "first",
    "days_since_prior_order": "first"
}).reset_index()

model_ready_cart = orders_features.merge(cart_size, on="order_id").dropna()

model_ready_cart.to_csv("data/model_ready_cart.csv", index=False)

print("Dataset Model 2 shape :", model_ready_cart.shape)

NameError: name 'df' is not defined

## Interpétation des résultats

Chaque observation correspond à une commande `(order_id)`. La variable cible cart_size représente le nombre total de produits dans le panier.

## Justification

Le dataset fusionné est transformé en deux datasets adaptés aux deux tâches de machine learning.
Pour la classification, l’unité d’observation est le produit dans une commande.
Pour la régression, les données sont agrégées au niveau de la commande afin de calculer la taille du panier.

# Model comparison

---

Les performances des modèles sont comparées afin d’identifier celui qui explique le mieux les comportements d’achat. Pour la classification, les métriques utilisées sont l’accuracy et le ROC-AUC. Pour la régression, les modèles sont évalués à l’aide du MAE et du R². Cette comparaison permet de déterminer si des modèles plus complexes apportent une amélioration par rapport aux modèles plus simples.

--

Deux types de modèles ont été utilisés dans ce projet afin de répondre aux deux tâches de machine learning. Pour la prédiction du **reorder**, une régression logistique et un Random Forest ont été entraînés. La régression logistique est un modèle simple qui permet d’établir une relation linéaire entre les variables et la probabilité de rachat d’un produit. Le Random Forest, en revanche, est capable de capturer des relations plus complexes entre les variables grâce à l’utilisation de plusieurs arbres de décision.

Pour la prédiction de la **taille du panier**, une régression linéaire et un Random Forest Regressor ont été utilisés. La régression linéaire fournit une estimation simple basée sur une relation linéaire entre les variables explicatives et la taille du panier. Le Random Forest Regressor permet quant à lui de modéliser des comportements d’achat plus complexes.

La comparaison des performances permet d’identifier le modèle le plus adapté pour chaque tâche. En général, les modèles Random Forest offrent de meilleures performances car ils peuvent capturer des relations non linéaires entre les variables.

# Conclusion

---

Ce projet a permis d’analyser les comportements d’achat des clients à partir du dataset NextBuy. L’analyse exploratoire a permis d’identifier les produits les plus populaires, les moments où les clients passent le plus de commandes et certaines relations entre les produits achetés ensemble.

Les modèles de machine learning montrent qu’il est possible de prédire certains comportements d’achat, comme la probabilité qu’un produit soit racheté ou la taille du panier d’une commande. Ces résultats permettent de mieux comprendre les habitudes des clients et peuvent être utilisés pour améliorer les recommandations produits ou optimiser les stratégies commerciales.